In [1]:
# ============================================================
# SELF-CONTAINED DIAGNOSTIC  (fresh session, no prior cells needed)
# ============================================================
import numpy as np
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/shauryapandey19/pm25-dataset/delhi_33station_pm25_spatiotemporal_dataset_2018_2026.csv"

use_columns = [
    "timestamp_ist", "station_name", "latitude", "longitude", "pm25", "pm25_original",
    "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos", "boundary_layer_height_m",
    "month", "fire_frp_sum", "fire_count",
]

df = pd.read_csv(DATA_PATH, usecols=use_columns, parse_dates=["timestamp_ist"])
df = df.rename(columns={"timestamp_ist": "timestamp"})
df["station_id"] = df["station_name"]
df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

STATION_COL = "station_id"   # guaranteed now
STATION     = "ihbas_dilshad_garden"

assert STATION in df[STATION_COL].unique(), f"{STATION} not in data"

print(f"df.shape={df.shape}  |  station column = {STATION_COL}")
print("=" * 70)
print("1. RAW DATA QUALITY  (z-scores vs cohort; |z|>2 = suspicious)")
print("=" * 70)

per_station = df.groupby(STATION_COL).agg(
    n_rows=("pm25", "size"),
    pm25_null=("pm25", lambda s: s.isna().mean()),
    pm25_orig_null=("pm25_original", lambda s: s.isna().mean()),
    pm25_mean=("pm25", "mean"),
    pm25_std=("pm25", "std"),
    pm25_max=("pm25", "max"),
    pm25_frac_at_min=("pm25", lambda s: (s == s.min()).mean()),
    blh_null=("boundary_layer_height_m", lambda s: s.isna().mean()),
)
srt = df.sort_values([STATION_COL, "timestamp"])
per_station["frac_flatline"] = srt.groupby(STATION_COL)["pm25"].apply(
    lambda s: (s.diff() == 0).mean()
)

print("\nAll stations sorted by frac_flatline:")
print(per_station.sort_values("frac_flatline", ascending=False).round(3).to_string())

print(f"\n>>> FOCUS: {STATION}")
print(per_station.loc[STATION].round(3).to_string())

print("\n>>> Z-scores vs cohort:")
for col in ["pm25_mean", "pm25_std", "frac_flatline", "pm25_frac_at_min",
            "pm25_orig_null", "pm25_null", "blh_null", "pm25_max"]:
    vals = per_station[col].astype(float)
    z = (vals.loc[STATION] - vals.mean()) / (vals.std() + 1e-9)
    flag = "  <<< SUSPICIOUS" if abs(z) > 2 else ""
    print(f"  {col:22s} z = {z:+.2f}{flag}")

print("\n" + "=" * 70)
print("2. TARGET DISTRIBUTION")
print("=" * 70)

station_df = df[df[STATION_COL] == STATION]
for name, d in [("THIS STATION", station_df), ("ALL STATIONS", df)]:
    y = d["pm25_original"].dropna()
    print(f"\n{name}  n_obs = {len(y):,}")
    print(f"  mean={y.mean():.2f}  std={y.std():.2f}  min={y.min():.2f}  max={y.max():.2f}")
    print("  quantiles: " + ", ".join(f"p{q}={y.quantile(q/100):.1f}" for q in [1,5,25,50,75,95,99]))

print(f"\n>>> THIS STATION  pm25_original >  500: {(station_df['pm25_original'] >  500).sum():,}")
print(f">>> THIS STATION  pm25_original > 1000: {(station_df['pm25_original'] > 1000).sum():,}")
print(f">>> ALL STATIONS  pm25_original >  500: {(df['pm25_original'] >  500).sum():,}")
print(f">>> ALL STATIONS  pm25_original > 1000: {(df['pm25_original'] > 1000).sum():,}")

print("\n" + "=" * 70)
print("3. THE STATION'S OWN ROWS")
print("=" * 70)
print(f"n rows = {len(station_df):,}")
print(f"date range: {station_df['timestamp'].min()} -> {station_df['timestamp'].max()}")
print(f"unique pm25 values: {station_df['pm25'].nunique():,}")
print(f"most common pm25 values:")
print(station_df['pm25'].value_counts().head(10).to_string())
print(f"\nlongest run of identical consecutive pm25 values:")
runs = (station_df.sort_values('timestamp')['pm25'].diff() != 0).cumsum()
run_lens = station_df.sort_values('timestamp').groupby(runs).size()
print(f"  {int(run_lens.max())} rows")

df.shape=(2392598, 14)  |  station column = station_id
1. RAW DATA QUALITY  (z-scores vs cohort; |z|>2 = suspicious)

All stations sorted by frac_flatline:
                                    n_rows  pm25_null  pm25_orig_null  pm25_mean  pm25_std  pm25_max  pm25_frac_at_min  blh_null  frac_flatline
station_id                                                                                                                                     
aurobindo_marg                       70903        0.0           0.005     82.391    78.915    992.75             0.000     0.061          0.039
alipur                               66457        0.0           0.010    101.364    94.374    992.00             0.001     0.065          0.037
najafgarh                            72415        0.0           0.016     81.920    75.678   1000.00             0.000     0.059          0.036
jawaharlal_nehru_stadium             73411        0.0           0.006     91.520    90.431   1000.00             0.000     0

In [3]:
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

DATA_PATH = "/kaggle/input/datasets/shauryapandey19/pm25-dataset/delhi_33station_pm25_spatiotemporal_dataset_2018_2026.csv"

# only the columns the model uses, keeps RAM and load time down on ~2.5M rows
use_columns = [
    "timestamp_ist", "station_name", "latitude", "longitude", "pm25", "pm25_original",
    "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos", "boundary_layer_height_m",
    "month", "fire_frp_sum", "fire_count",
]

df = pd.read_csv(DATA_PATH, usecols=use_columns, parse_dates=["timestamp_ist"])

df = df.rename(columns={"timestamp_ist": "timestamp"})
df["station_id"] = df["station_name"]
df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

print(df.shape)
print(df["station_id"].nunique())

# must be 0, the pivot in cell 4 needs unique (station, hour) pairs
print(df.duplicated(subset=["station_id", "timestamp"]).sum())

(2392598, 14)
33
0


In [4]:
# prep already set PM2.5 <= 0 to NaN before interpolation, so this only verifies it
print("pm25_original <= 0:", int((df["pm25_original"] <= 0).sum()))
print("pm25 <= 0:", int((df["pm25"] <= 0).sum()), "| pm25 nulls:", int(df["pm25"].isna().sum()))
print("pm25_original nulls (interpolated hours):", int(df["pm25_original"].isna().sum()))

assert df["pm25"].notna().all() and (df["pm25"] > 0).all()
assert not (df["pm25_original"] <= 0).any()  # NaN compares False, so nulls pass

pm25_original <= 0: 0
pm25 <= 0: 0 | pm25 nulls: 0
pm25_original nulls (interpolated hours): 25674


In [16]:
# makes the cell safe to re-run: removes lag columns from an earlier run, including the _x / _y copies a repeated merge creates
df = df.drop(columns=[c for c in df.columns if c.startswith("fire_") and "_lag" in c])

df["ist_date"] = df["timestamp"].dt.normalize()

# fire columns are one value per (station, IST day), repeated on every hour of that day
fire_daily_columns = ["fire_frp_sum", "fire_count"]
assert df.groupby(["station_id", "ist_date"])[fire_daily_columns].nunique().to_numpy().max() == 1

daily_fire = df.drop_duplicates(["station_id", "ist_date"])[["station_id", "ist_date"] + fire_daily_columns]

for lag_days in [1, 2, 3]:
    lag_table = daily_fire.copy()
    # shifting the date forward puts day D's value on day D + lag_days
    lag_table["ist_date"] = lag_table["ist_date"] + pd.Timedelta(days=lag_days)
    lag_table = lag_table.rename(columns={
        "fire_frp_sum": "fire_frp_lag" + str(lag_days),
        "fire_count": "fire_count_lag" + str(lag_days),
    })
    df = df.merge(lag_table, on=["station_id", "ist_date"], how="left")

df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

# NaN lags = first days of 2018 or a missing previous day, LightGBM takes NaN natively
fire_source_columns = [c for c in df.columns if c.startswith("fire_frp") or c.startswith("fire_count")]
print(df[fire_source_columns].isna().sum())
assert (df[fire_source_columns].min() >= 0).all()

# log1p because FRP and counts are very skewed
df["fire_frp_today_log"] = np.log1p(df["fire_frp_sum"])
df["fire_frp_past3_log"] = np.log1p((df["fire_frp_lag1"] + df["fire_frp_lag2"] + df["fire_frp_lag3"]) / 3)
df["fire_count_today_log"] = np.log1p(df["fire_count"])
df["fire_count_past3_log"] = np.log1p((df["fire_count_lag1"] + df["fire_count_lag2"] + df["fire_count_lag3"]) / 3)

fire_count                  0
fire_frp_sum                0
fire_frp_today_log          0
fire_frp_past3_log      25900
fire_count_today_log        0
fire_count_past3_log    25900
fire_frp_lag1            5267
fire_count_lag1          5267
fire_frp_lag2           13129
fire_count_lag2         13129
fire_frp_lag3           17253
fire_count_lag3         17253
dtype: int64


In [17]:
LAG_HOURS = [1, 3, 6]
NEIGHBOR_K = 3

station_list = sorted(df["station_id"].unique())
n_stations = len(station_list)
station_to_idx = {station: i for i, station in enumerate(station_list)}
target_idx = df["station_id"].map(station_to_idx).to_numpy()
n_rows = len(df)

# complete hourly index, so shifting the grid by n rows is exactly n hours
full_index = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")

# row -> position of its hour in the grid
time_idx = full_index.get_indexer(df["timestamp"])
assert (time_idx >= 0).all()

# pm25_grid[t, i] = pm25 of station i at hour t, NaN where the station has no row
pm25_grid = (
    df.pivot(index="timestamp", columns="station_id", values="pm25")
      .reindex(index=full_index, columns=station_list)
      .to_numpy(dtype=np.float32)
)

# value_grids[0] = pm25 now, then 1h, 3h, 6h ago
value_grids = [pm25_grid]
for lag in LAG_HOURS:
    shifted = np.full_like(pm25_grid, np.nan)
    shifted[lag:] = pm25_grid[:-lag]
    # a lagged value only counts where the station also has a row at hour t,
    # same as merging lag columns onto existing rows in the shared notebook
    shifted[np.isnan(pm25_grid)] = np.nan
    value_grids.append(shifted)

# availability per grid and number of stations reporting per hour, used for the comparable-row rule
avail_grids = [~np.isnan(g) for g in value_grids]
avail_count = [a.sum(axis=1) for a in avail_grids]

# trailing 24h mean of each station's pm25, current hour included, never future hours.
# grid rows are consecutive hours, so rolling(24) is exactly 24 hours
ROLL_HOURS = 24
roll_grid = (
    pd.DataFrame(pm25_grid)
    .rolling(ROLL_HOURS, min_periods=ROLL_HOURS // 2)   # needs at least 12 real hours in the window
    .mean()
    .to_numpy(dtype=np.float32)
)

print(n_stations, n_rows, pm25_grid.shape)

33 2392598 (76680, 33)


In [18]:
station_coords = df.drop_duplicates("station_id").set_index("station_id").loc[station_list, ["latitude", "longitude"]]
lat_r = np.radians(station_coords["latitude"].to_numpy())
lon_r = np.radians(station_coords["longitude"].to_numpy())

# dist_matrix[j, i] = haversine km from source station i to target station j (broadcast, no loop)
a_term = (np.sin((lat_r[None, :] - lat_r[:, None]) / 2) ** 2
          + np.cos(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.sin((lon_r[None, :] - lon_r[:, None]) / 2) ** 2)
dist_matrix = 2 * 6371.0 * np.arcsin(np.sqrt(a_term))

# (n_rows, n_stations) distance from each row's station to every source, built once and reused by every fold.
# inf marks sources that can never be a neighbour for that row
dist_avail = dist_matrix[target_idx].astype(np.float32)
dist_avail[np.arange(n_rows), target_idx] = np.inf   # own station
dist_avail[~avail_grids[0][time_idx]] = np.inf       # source has no pm25 at that hour

In [19]:
IDW_POWER = 1.5       # K(distance) = distance ** -IDW_POWER
WIND_KAPPA = 1.0      # G(alignment) = exp(WIND_KAPPA * alignment), upwind weighs e**2 times more than downwind
MIN_DIST_KM = 0.5     # floor, so stations with near-identical coordinates cannot get infinite weight

# initial great-circle bearing from target station j to source station i, radians clockwise from north.
# axis 0 = target, axis 1 = source, same convention as dist_matrix
d_lon = lon_r[None, :] - lon_r[:, None]
bearing = np.arctan2(
    np.sin(d_lon) * np.cos(lat_r[None, :]),
    np.cos(lat_r[:, None]) * np.sin(lat_r[None, :])
    - np.sin(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.cos(d_lon),
)
bearing_sin = np.sin(bearing).astype(np.float32)
bearing_cos = np.cos(bearing).astype(np.float32)

# wind_dir is the direction the wind comes FROM, so a source lying in that direction is upwind.
# alignment = cos(bearing - wind_from) = sin*sin + cos*cos, +1 directly upwind, -1 directly downwind.
# fancy index bearing_sin[target_idx] gives one (n_stations,) row per df row -> (n_rows, n_stations)
wind_sin = df["wind_dir_sin"].to_numpy(np.float32)[:, None]
wind_cos = df["wind_dir_cos"].to_numpy(np.float32)[:, None]
alignment = wind_sin * bearing_sin[target_idx] + wind_cos * bearing_cos[target_idx]

# w_i = K(distance_i) * G(alignment_i), for every source station at every row
dist_floor = np.maximum(dist_matrix, MIN_DIST_KM).astype(np.float32)
w_static = np.exp(WIND_KAPPA * alignment) / dist_floor[target_idx] ** IDW_POWER
del alignment

# a source with no pm25 at that hour or the row's own station gets zero weight
w_static[~avail_grids[0][time_idx]] = 0.0
w_static[np.arange(n_rows), target_idx] = 0.0
assert (w_static[np.arange(n_rows), target_idx] == 0).all()

# source pm25 at each row's hour, NaN zero-filled (safe, those weights are already 0)
pm25_rows = np.nan_to_num(value_grids[0][time_idx], nan=0.0)

# full-network weighted sums in float64. a fold later subtracts the held-out station's column
# instead of recomputing the (n_rows, n_stations) products 33 times
wsum_full = w_static.sum(axis=1, dtype=np.float64)
wpm_full = (w_static * pm25_rows).sum(axis=1, dtype=np.float64)

In [20]:
def build_neighbor_features(held_idx=None):

    dist_rows = dist_avail.copy()

    # the held-out station is removed from the network for every row, train and test
    if held_idx is not None:
        dist_rows[:, held_idx] = np.inf

    # K smallest distances per row via argpartition (no full sort). the order inside the K is arbitrary,
    # so "nearest" is found explicitly below, and mean/min/max/std do not depend on order
    order = np.argpartition(dist_rows, NEIGHBOR_K - 1, axis=1)[:, :NEIGHBOR_K]
    sel_dist = np.take_along_axis(dist_rows, order, axis=1)
    is_valid = np.isfinite(sel_dist)

    # a valid pick can never be the row's own station or the held-out one
    assert not np.any((order == target_idx[:, None]) & is_valid)
    if held_idx is not None:
        assert not np.any((order == held_idx) & is_valid)

    n_valid = is_valid.sum(axis=1)
    time_col = time_idx[:, None]     # (n_rows, 1), broadcasts against order (n_rows, K) to gather grid[t, station]
    row_col = target_idx[:, None]
    result = {}

    with warnings.catch_warnings():
        # all-NaN rows are expected, they just become NaN features
        warnings.simplefilter("ignore", category=RuntimeWarning)

        # pm25 now: mean, min, max, spread of the selected neighbours
        now_vals = value_grids[0][time_col, order].astype(np.float64)
        now_vals[~is_valid] = np.nan

        result["neighbor_mean"] = np.nanmean(now_vals, axis=1)
        result["neighbor_min"] = np.nanmin(now_vals, axis=1)
        result["neighbor_max"] = np.nanmax(now_vals, axis=1)

        # std needs at least 2 neighbours
        neighbor_std = np.nanstd(now_vals, axis=1)
        neighbor_std[n_valid < 2] = np.nan
        result["neighbor_std"] = neighbor_std

        # same neighbours, pm25 1h, 3h, 6h ago
        lag_names = ["neighbor_mean_lag1", "neighbor_mean_lag3", "neighbor_mean_lag6"]
        lag_arrays = []
        for k in range(3):
            lag_vals = value_grids[k + 1][time_col, order].astype(np.float64)
            lag_vals[~is_valid] = np.nan
            lag_arrays.append(lag_vals)
            result[lag_names[k]] = np.nanmean(lag_vals, axis=1)

        # recent change: per-neighbour difference first, then the mean, so only neighbours that have
        # BOTH values count (a difference of two means over different neighbour sets would be wrong)
        result["neighbor_change_1h"] = np.nanmean(now_vals - lag_arrays[0], axis=1)
        result["neighbor_change_3h"] = np.nanmean(now_vals - lag_arrays[1], axis=1)

        # trailing 24h mean of the same neighbours
        roll_vals = roll_grid[time_col, order].astype(np.float64)
        roll_vals[~is_valid] = np.nan
        result["neighbor_mean_roll24"] = np.nanmean(roll_vals, axis=1)

        # distance from the target to its neighbours
        dist_vals = sel_dist.astype(np.float64)
        dist_vals[~is_valid] = np.nan
        result["nearest_dist_km"] = np.nanmin(dist_vals, axis=1)
        result["mean_dist_km"] = np.nanmean(dist_vals, axis=1)

        # bearing from the target to its neighbours, (n_rows, K) gather from the (target, source) matrices
        b_sin = bearing_sin[row_col, order].astype(np.float64)
        b_cos = bearing_cos[row_col, order].astype(np.float64)
        b_sin[~is_valid] = np.nan
        b_cos[~is_valid] = np.nan

        # mean of unit vectors = resultant vector: direction says where the neighbours sit,
        # length (shorter when they surround the target) says how spread out they are
        result["mean_bearing_sin"] = np.nanmean(b_sin, axis=1)
        result["mean_bearing_cos"] = np.nanmean(b_cos, axis=1)

        # nearest neighbour's own bearing. argmin over sel_dist picks it (inf for invalid picks)
        nearest_pos = np.argmin(sel_dist, axis=1)
        rows_range = np.arange(n_rows)
        result["nearest_bearing_sin"] = b_sin[rows_range, nearest_pos]
        result["nearest_bearing_cos"] = b_cos[rows_range, nearest_pos]

        # wind-weighted mean over ALL available stations, normalised by the weight sum.
        # held-out station's contribution is subtracted from the precomputed full-network sums
        if held_idx is None:
            w_den = wsum_full
            w_num = wpm_full
        else:
            w_den = wsum_full - w_static[:, held_idx].astype(np.float64)
            w_num = wpm_full - (w_static[:, held_idx] * pm25_rows[:, held_idx]).astype(np.float64)

        # if only the held-out station had weight, the residual is rounding noise, so it becomes NaN
        w_ok = w_den > 1e-6
        result["neighbor_wmean"] = np.where(w_ok, w_num / np.where(w_ok, w_den, 1.0), np.nan)

    return result

# quick test, hold out station 0 and look at the missing counts
test_feats = build_neighbor_features(0)
for name in test_feats:
    print(name, np.isnan(test_feats[name]).sum())

neighbor_mean 0
neighbor_min 0
neighbor_max 0
neighbor_std 0
neighbor_mean_lag1 121
neighbor_mean_lag3 423
neighbor_mean_lag6 890
neighbor_change_1h 121
neighbor_change_3h 423
neighbor_mean_roll24 1154
nearest_dist_km 0
mean_dist_km 0
mean_bearing_sin 0
mean_bearing_cos 0
nearest_bearing_sin 0
nearest_bearing_cos 0
neighbor_wmean 0


In [21]:
# ============================================================
# PHYSICS CONSTANTS  (all starting guesses, tune after the first run)
# ============================================================
WIND_FLOOR_MS = 0.5          # m/s, lowest wind speed used; stops travel time = distance / 0 = infinity on calm hours
TAU_MAX_H = 6.0              # hours, a source whose air needs longer than this to arrive gets weight 0 (hourly wind is not trusted that long)
SIGMA_BASE_DEG = 20.0        # deg, minimum half-width of the upwind cone (wind direction measurement noise)
SIGMA_DIST_DEG_PER_KM = 2.0  # deg per km, cone widens with distance (plume spreads sideways as it travels)
SIGMA_CALM_DEG = 30.0        # deg*(m/s), cone widens on weak wind by SIGMA_CALM_DEG / speed (calm air is nearly directionless)
SIGMA_MAX_DEG = 90.0         # deg, cap on the cone half-width
T_FADE_H = 2.0               # hours, e-folding time: after T_FADE_H hours of travel the weight is down to 1/e
CHUNK_ROWS = 250_000         # rows per block, keeps the temporary (rows x stations) arrays small in RAM

# wind speed per row as a column vector (n_rows, 1). nan_to_num turns NaN into 0, then maximum lifts everything below the floor up to the floor
speed_all = np.maximum(np.nan_to_num(df["wind_speed_m_s"].to_numpy(np.float32), nan=0.0), WIND_FLOOR_MS)[:, None]

# row vector 0..n_stations-1, used with fancy indexing to read "station j at hour h" from pm25_grid for all j at once
src_cols = np.arange(n_stations)[None, :]

# outputs, one value per (row, source station). all zero = "this source carries no information for this row"
w_phys = np.zeros((n_rows, n_stations), dtype=np.float32)      # final weight w_ij = cone weight * fade weight (0 if invalid)
adv_phys = np.zeros((n_rows, n_stations), dtype=np.float32)    # source PM2.5 at the moment the arriving air left it: pm_j(t - tau)
trend_phys = np.zeros((n_rows, n_stations), dtype=np.float32)  # pm_j(t) - pm_j(t - tau): did the source rise or fall during the trip
tau_phys = np.zeros((n_rows, n_stations), dtype=np.float32)    # travel time in hours from source j to the row's station

for r0 in range(0, n_rows, CHUNK_ROWS):
    r1 = min(r0 + CHUNK_ROWS, n_rows)          # last row of this block (exclusive)
    n_c = r1 - r0                              # rows in this block
    tgt = target_idx[r0:r1]                    # (n_c,) station index of each row (the target i)
    t_c = time_idx[r0:r1]                      # (n_c,) hour index of each row in the hourly grid
    d_km = dist_floor[tgt]                     # (n_c, n_stations) distance target -> every source in km, floored at MIN_DIST_KM
    speed = speed_all[r0:r1]                   # (n_c, 1) wind speed in m/s

    # ---- 1) direction match: angle between "where the wind comes from" and "where the source sits" ----
    # cos(bearing - wind_from) = sin*sin + cos*cos, +1 = source exactly upwind, -1 = exactly downwind
    align = wind_sin[r0:r1] * bearing_sin[tgt] + wind_cos[r0:r1] * bearing_cos[tgt]
    # delta in radians, 0 = directly upwind, pi = directly downwind. clip guards arccos against float rounding like 1.0000001
    delta = np.arccos(np.clip(align, -1.0, 1.0))

    # ---- 2) travel time ----
    # d [km] / (speed [m/s] * 3.6 [km/h per m/s]) = hours
    tau_h = d_km / (3.6 * speed)

    # ---- 3) cone half-width sigma (degrees -> radians), wider when far away or when wind is weak ----
    sigma_deg = np.minimum(SIGMA_BASE_DEG + SIGMA_DIST_DEG_PER_KM * d_km + SIGMA_CALM_DEG / speed, SIGMA_MAX_DEG)
    sigma_rad = np.radians(sigma_deg)

    # Gaussian in the angle: 1 on the wind axis, 0.61 at delta = sigma, 0.14 at delta = 2*sigma
    w_cone = np.exp(-(delta ** 2) / (2.0 * sigma_rad ** 2))

    # ---- 4) fade with travel time: longer trip = more mixing and change on the way = less trust ----
    w_fade = np.exp(-tau_h / T_FADE_H)

    # combined weight of source j for target i
    w = w_cone * w_fade

    # ---- 5) source PM2.5 at the moment the air left it, linear interpolation between two hourly values ----
    k_h = np.floor(tau_h).astype(np.int32)      # whole hours of the trip
    frac = (tau_h - k_h).astype(np.float32)     # leftover fraction of an hour, in [0, 1)
    i0 = t_c[:, None] - k_h                     # grid index of hour (t - k): the newer of the two hours
    i1 = i0 - 1                                 # grid index of hour (t - k - 1): the older of the two hours
    # a source is usable only if the trip is short enough and both hours exist inside the grid
    time_ok = (tau_h <= TAU_MAX_H) & (i1 >= 0)
    # clip negative indices so the gather below never goes out of bounds, those entries are discarded through time_ok
    i0 = np.clip(i0, 0, None)
    i1 = np.clip(i1, 0, None)

    # gather PM2.5 of every source at three hours: now, newer hour, older hour. all shapes (n_c, n_stations)
    p_now = pm25_grid[t_c[:, None], src_cols]
    p_0 = pm25_grid[i0, src_cols]
    p_1 = pm25_grid[i1, src_cols]

    # pm_j(t - tau) = (1 - frac) * pm(t - k) + frac * pm(t - k - 1). frac = 0 gives exactly pm(t - k), frac -> 1 gives pm(t - k - 1)
    p_adv = (1.0 - frac) * p_0 + frac * p_1
    # change of the source over the trip: positive = the source got dirtier since the air left it
    p_trend = p_now - p_adv

    # ---- 6) validity: one mask for all outputs so every feature uses exactly the same set of sources ----
    valid = time_ok & np.isfinite(p_now) & np.isfinite(p_0) & np.isfinite(p_1)
    valid[np.arange(n_c), tgt] = False          # a station never predicts itself (its own bearing is undefined)

    # write the block. invalid entries become 0, so they drop out of every weighted sum
    w_phys[r0:r1] = np.where(valid, np.nan_to_num(w, nan=0.0), 0.0)
    adv_phys[r0:r1] = np.where(valid, p_adv, 0.0)
    trend_phys[r0:r1] = np.where(valid, p_trend, 0.0)
    tau_phys[r0:r1] = tau_h

# full-network weighted sums in float64 (sum over ALL 33 sources). a fold later subtracts the held-out column
wsum_phys_full = w_phys.sum(axis=1, dtype=np.float64)                     # sum of w_ij
wadv_phys_full = (w_phys * adv_phys).sum(axis=1, dtype=np.float64)        # sum of w_ij * pm_j(t - tau_ij)
wtr_phys_full = (w_phys * trend_phys).sum(axis=1, dtype=np.float64)       # sum of w_ij * (pm_j(t) - pm_j(t - tau_ij))

# checks: own station carries no weight, and how many sources typically matter
assert (w_phys[np.arange(n_rows), target_idx] == 0).all()
print("rows with at least one usable upwind source:", f"{(wsum_phys_full > 1e-9).mean():.3f}")
print("average number of sources with weight > 0.01:", f"{(w_phys > 0.01).sum(axis=1).mean():.2f}")

rows with at least one usable upwind source: 1.000
average number of sources with weight > 0.01: 25.77


In [22]:
PHYSICS_MIN_WEIGHT = 1e-9   # below this the weight sum is rounding noise (or no usable source), so the features become NaN

def build_physics_features(held_idx=None):

    rows_all = np.arange(n_rows)

    # start from the full-network sums, copies because the held-out subtraction below is in place
    w_den = wsum_phys_full.copy()          # sum_j w_ij
    w_num_adv = wadv_phys_full.copy()      # sum_j w_ij * adv_ij
    w_num_tr = wtr_phys_full.copy()        # sum_j w_ij * trend_ij

    saved_col = None
    if held_idx is not None:
        # remove the held-out station from the network for every row (train and test), exact because
        # the weights of one source do not depend on the other sources
        held_w = w_phys[:, held_idx]
        w_den -= held_w.astype(np.float64)
        w_num_adv -= (held_w * adv_phys[:, held_idx]).astype(np.float64)
        w_num_tr -= (held_w * trend_phys[:, held_idx]).astype(np.float64)
        # held_w is a view, so copy it BEFORE overwriting the column
        saved_col = held_w.copy()
        # -1 makes sure argmax below can never pick the held-out station (valid weights are >= 0)
        w_phys[:, held_idx] = -1.0

    try:
        # index of the strongest upwind source per row, and its weight
        top_pos = np.argmax(w_phys, axis=1)
        top_w = w_phys[rows_all, top_pos]
    finally:
        # always restore the column, even if something above fails, otherwise later folds would be corrupted
        if held_idx is not None:
            w_phys[:, held_idx] = saved_col

    # rows where nothing usable is left become NaN (LightGBM reads NaN natively)
    w_ok = w_den > PHYSICS_MIN_WEIGHT
    safe_den = np.where(w_ok, w_den, 1.0)  # avoids 0/0 warnings, those rows are overwritten with NaN right after

    result = {
        # weighted mean of the upwind sources' PM2.5 at the time the air left them
        "phys_adv_pm25": np.where(w_ok, w_num_adv / safe_den, np.nan),
        # weighted mean of how much those sources changed during the trip (rising or falling pollution upwind)
        "phys_adv_trend": np.where(w_ok, w_num_tr / safe_den, np.nan),
        # total weight = how well the upwind signal is covered, 0 means no real upwind neighbour
        "phys_w_sum": np.where(w_ok, w_den, 0.0),
        # travel time in hours from the single strongest upwind source
        "phys_top_tau_h": np.where(top_w > PHYSICS_MIN_WEIGHT, tau_phys[rows_all, top_pos], np.nan),
    }
    return result

# ---- test: hold out station 0 and check against a slow brute-force calculation on 2000 random rows ----
test_phys = build_physics_features(0)
for name, arr in test_phys.items():
    print(name, "NaN:", int(np.isnan(arr).sum()), "| min:", np.nanmin(arr), "| max:", np.nanmax(arr))

assert (w_phys[:, 0] >= 0).all()   # the held-out column was restored

chk_rows = np.random.default_rng(0).choice(n_rows, 2000, replace=False)
w_chk = w_phys[chk_rows].astype(np.float64)
w_chk[:, 0] = 0.0                                                   # drop station 0 explicitly
den_chk = w_chk.sum(axis=1)
num_chk = (w_chk * adv_phys[chk_rows]).sum(axis=1)
ok_chk = den_chk > PHYSICS_MIN_WEIGHT
assert np.allclose(test_phys["phys_adv_pm25"][chk_rows][ok_chk], (num_chk / np.where(ok_chk, den_chk, 1.0))[ok_chk], rtol=1e-4)
print("physics feature check passed")

phys_adv_pm25 NaN: 108 | min: 2.2911613499448116 | max: 922.7480710865697
phys_adv_trend NaN: 108 | min: -451.8267538615721 | max: 625.8976862408647
phys_w_sum NaN: 0 | min: 0.0 | max: 25.652835190296173
phys_top_tau_h NaN: 108 | min: 0.009724516 | max: 5.9570527
physics feature check passed


In [25]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek

# Delhi seasons from month: winter 12-2, summer 3-6, monsoon 7-9, post-monsoon 10-11
df["season_winter"] = df["month"].isin([12, 1, 2]).astype(int)
df["season_summer"] = df["month"].between(3, 6).astype(int)
df["season_monsoon"] = df["month"].between(7, 9).astype(int)
df["season_post_monsoon"] = df["month"].isin([10, 11]).astype(int)

# wind as a vector. wind_dir is where the wind comes FROM, so the air moves toward direction + 180 deg:
# u = eastward component, v = northward component (this is the form a "wind what-if" will change later)
df["wind_u_ms"] = -df["wind_speed_m_s"] * df["wind_dir_sin"]
df["wind_v_ms"] = -df["wind_speed_m_s"] * df["wind_dir_cos"]

base_feature_columns = ["wind_speed_m_s", "wind_dir_sin", "wind_dir_cos", "wind_u_ms", 
    "wind_v_ms", "boundary_layer_height_m",
    "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos", "boundary_layer_height_m",
    "hour", "day_of_week",
    "season_winter", "season_summer", "season_monsoon", "season_post_monsoon",
    "latitude", "longitude"]

physics_feature_columns = ["phys_adv_pm25", "phys_adv_trend", "phys_w_sum", "phys_top_tau_h"]

neighbor_feature_columns = [
    "neighbor_mean", "neighbor_min", "neighbor_max", "neighbor_std",
    "neighbor_mean_lag1", "neighbor_mean_lag3", "neighbor_mean_lag6",
    "neighbor_mean_roll24", "neighbor_change_1h", "neighbor_change_3h",
    "neighbor_wmean",
    "nearest_dist_km", "mean_dist_km",
    "nearest_bearing_sin", "nearest_bearing_cos", "mean_bearing_sin", "mean_bearing_cos"
] + physics_feature_columns   # these 4 depend on the held-out station, so they are rebuilt every fold like the other neighbour columns

feature_columns = base_feature_columns + neighbor_feature_columns

# fire columns stay LAST, the no-fire model is the first n_base_features columns of X_all
fire_feature_columns = ["fire_frp_today_log", "fire_frp_past3_log", "fire_count_today_log", "fire_count_past3_log"]
feature_columns_fire = feature_columns + fire_feature_columns

# the fire version is the no-fire columns plus 2 at the end, so no-fire = first n_base_features columns
n_base_features = len(feature_columns)

# pm25_original is the real target, pm25 (interpolated) is only ever an input
target_column = "pm25_original"

# boundary_layer_height_m is null for 2024-01-01 to 2024-06-30, so those rows are excluded here (same rule as the shared notebook)
required_columns = ["pm25_original", "boundary_layer_height_m", "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos"]
row_ok = df[required_columns].notna().all(axis=1).to_numpy()

# one float32 matrix for all models. neighbour columns are placeholders, overwritten in place for every fold
X_all = np.empty((n_rows, len(feature_columns_fire)), dtype=np.float32)
for pos, col in enumerate(feature_columns_fire):
    if col not in neighbor_feature_columns:
        X_all[:, pos] = df[col].to_numpy()
y_all = df[target_column].to_numpy(dtype=np.float64)

neighbor_pos = {name: feature_columns_fire.index(name) for name in neighbor_feature_columns}

def set_neighbor_columns(held_idx):
    feats = build_neighbor_features(held_idx)
    feats.update(build_physics_features(held_idx))   # adds the 4 physics columns to the same dict
    for name, pos in neighbor_pos.items():
        X_all[:, pos] = feats[name]
        
# the GAM only predicts a row if some other station has pm25 at t, t-1h, t-3h and t-6h
def comparable_rows(held_idx):
    keep = np.ones(n_rows, dtype=bool)
    for avail, count in zip(avail_grids, avail_count):
        # stations other than the held-out one that have a value at each hour, then looked up per row
        other_sources = count - avail[:, held_idx].astype(np.int64)
        keep &= other_sources[time_idx] > 0
    return keep

print(len(feature_columns), len(feature_columns_fire), int(row_ok.sum()), n_rows)

39 43 2230464 2392598


In [26]:
lgbm_best_params = {'subsample': 0.9, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 40,
                    'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 1.0}

# cell 9 reads lgbm_random_search.best_params_, so this stand-in keeps it working unchanged
lgbm_random_search = type("Fixed", (), {"best_params_": lgbm_best_params})()

In [27]:
def calculate_metrics(actual_values, predicted_values):

    # float64 so sums over millions of rows do not lose precision
    actual_values = np.asarray(actual_values, dtype=np.float64)
    predicted_values = np.asarray(predicted_values, dtype=np.float64)
    errors = actual_values - predicted_values

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))

    ss_residual = np.sum(errors ** 2)
    ss_total = np.sum((actual_values - np.mean(actual_values)) ** 2)
    r2 = 1 - (ss_residual / ss_total)

    # positive means over-prediction
    bias = np.mean(predicted_values - actual_values)

    return mae, rmse, r2, bias

lgbm_best_params = lgbm_random_search.best_params_

def fit_and_predict(train_mask, test_mask):

    # boolean-mask indexing copies once per fold, both versions then slice columns from these copies
    X_train_full = X_all[train_mask]
    y_train = y_all[train_mask]
    X_test_full = X_all[test_mask]

    predictions = {}

    # same rows for both versions, only the number of columns differs
    for tag, n_cols in [("", n_base_features), ("_fire", len(feature_columns_fire))]:
        lgbm_model = lgb.LGBMRegressor(
            random_state=42, verbosity=-1, n_jobs=-1, force_col_wise=True, subsample_freq=1,
            **lgbm_best_params
        )
        # column slice is a non-contiguous view, make it contiguous once instead of letting LightGBM copy it
        lgbm_model.fit(np.ascontiguousarray(X_train_full[:, :n_cols]), y_train)
        predictions["lgbm" + tag + "_predicted"] = lgbm_model.predict(np.ascontiguousarray(X_test_full[:, :n_cols]))

    return predictions

timestamps = df["timestamp"].to_numpy()

# one DataFrame per fold, concatenated at the end (no per-row Python loop)
def make_result_frame(station_name, test_mask, predictions):
    frame = pd.DataFrame({
        "station_id": station_name,
        "timestamp": timestamps[test_mask],
        "actual": y_all[test_mask],
    })
    for model_name, values in predictions.items():
        frame[model_name] = values
    return frame

In [28]:
# time block for the date-leakage check: autumn stubble season (IST)
block_start = pd.Timestamp("2025-10-15")
block_end = pd.Timestamp("2025-12-01")
in_block = ((df["timestamp"] >= block_start) & (df["timestamp"] < block_end)).to_numpy()
print("rows inside the time block:", int(in_block.sum()))

main_frames = []
block_frames = []

for held_idx in range(n_stations):

    t_start = time.perf_counter()
    held_out_station = station_list[held_idx]

    # neighbours for every row come from the other stations only
    set_neighbor_columns(held_idx)

    is_held = target_idx == held_idx
    comparable = comparable_rows(held_idx)

    # standard LOSO: all hours of the other stations are available for training
    train_mask = row_ok & (~is_held)
    test_mask = row_ok & is_held & comparable

    if test_mask.sum() == 0:
        print(held_out_station, "skipped, no test rows")
        continue

    predictions = fit_and_predict(train_mask, test_mask)
    main_frames.append(make_result_frame(held_out_station, test_mask, predictions))

    # block LOSO: also remove the stubble-season hours of ALL stations from training
    block_train_mask = train_mask & (~in_block)
    block_test_mask = test_mask & in_block

    if block_test_mask.sum() > 0:
        block_predictions = fit_and_predict(block_train_mask, block_test_mask)
        block_frames.append(make_result_frame(held_out_station, block_test_mask, block_predictions))

    print(held_out_station, int(test_mask.sum()), int(block_test_mask.sum()), f"{time.perf_counter() - t_start:.0f}s")

loso_df = pd.concat(main_frames, ignore_index=True)
block_df = pd.concat(block_frames, ignore_index=True)

loso_df.to_csv("/kaggle/working/loso_df_lgbm_fire_33station.csv", index=False)
block_df.to_csv("/kaggle/working/block_df_lgbm_fire_33station.csv", index=False)

print(loso_df.shape, block_df.shape)

rows inside the time block: 36723
alipur 61510 1106 163s
anand_vihar 65681 1117 151s
ashok_vihar 68243 1116 151s
aurobindo_marg 66207 1120 149s
aya_nagar 66874 1124 153s
bawana 65523 1119 151s
crri_mathura_road 68115 1111 151s
dr_karni_singh_shooting_range 66524 1124 150s
dtu 66459 1088 150s
igi_airport_t3 65714 1122 150s
ihbas_dilshad_garden 67578 1094 153s
ito 66689 1102 149s
jahangirpuri 68557 1101 152s
jawaharlal_nehru_stadium 68608 1103 149s
major_dhyan_chand_national_stadium 69664 1117 152s
mandir_marg 67385 1094 150s
mundka 65579 1111 153s
najafgarh 67058 1029 150s
narela 69499 1128 153s
nehru_nagar 69743 1117 153s
north_campus_du 67047 1123 152s
nsut 69256 1079 150s
okhla_phase_2 69820 1071 152s
patparganj 69210 1118 153s
punjabi_bagh 68883 1004 151s
pusa 64896 1114 152s
rk_puram 67169 1109 152s
rohini 69895 1116 149s
shadipur 68763 1110 151s
sirifort 67754 1098 152s
sonia_vihar 68868 1096 152s
vivek_vihar 68371 1124 152s
wazirpur 68852 1121 153s
(2229994, 5) (36426, 5)


In [30]:
model_columns = ["lgbm_predicted", "lgbm_fire_predicted"]

for col in model_columns:
    mae, rmse, r2, bias = calculate_metrics(loso_df["actual"], loso_df[col])
    print(f"{col:22s} MAE {mae:.3f}  RMSE {rmse:.2f}  R2 {r2:.3f}  bias {bias:.2f}")

per_station_rows = []

for station, station_rows in loso_df.groupby("station_id"):

    row = {"station_id": station, "n_rows": len(station_rows)}

    for col in model_columns:
        tag = col.replace("_predicted", "")
        mae, rmse, r2, bias = calculate_metrics(station_rows["actual"], station_rows[col])
        row[tag + "_mae"] = mae
        row[tag + "_rmse"] = rmse
        row[tag + "_r2"] = r2
        row[tag + "_bias"] = bias

    per_station_rows.append(row)

per_station_df = pd.DataFrame(per_station_rows)
print(per_station_df.round(3).to_string())

print("fire lowers MAE at", int((per_station_df["lgbm_fire_mae"] < per_station_df["lgbm_mae"]).sum()),
      "of", len(per_station_df), "stations")

lgbm_predicted         MAE 24.692  RMSE 42.87  R2 0.809  bias 1.14
lgbm_fire_predicted    MAE 24.654  RMSE 42.85  R2 0.809  bias 0.96
                            station_id  n_rows  lgbm_mae  lgbm_rmse  lgbm_r2  lgbm_bias  lgbm_fire_mae  lgbm_fire_rmse  lgbm_fire_r2  lgbm_fire_bias
0                               alipur   61510    21.290     36.032    0.859      9.919         21.107          35.144         0.866           9.891
1                          anand_vihar   65681    29.318     49.779    0.821    -23.310         28.907          49.255         0.824         -22.508
2                          ashok_vihar   68243    23.940     37.864    0.877     17.528         23.322          36.951         0.882          16.569
3                       aurobindo_marg   66207    17.381     29.809    0.856      9.422         17.373          29.999         0.854           9.436
4                            aya_nagar   66874    24.592     38.978    0.722      7.574         24.489          39.175   

In [31]:
ts = loso_df["timestamp"]
window_mask = (ts >= block_start) & (ts < block_end)

subsets = [
    ("all rows", loso_df),
    ("actual < 100", loso_df[loso_df["actual"] < 100]),
    ("actual >= 100", loso_df[loso_df["actual"] >= 100]),
    ("Oct 15-Nov 30 2025, normal LOSO", loso_df[window_mask]),
    ("Oct 15-Nov 30 2025, window removed from training", block_df)
]

for label, data in subsets:
    print("-----", label, "| n =", len(data))
    for col in model_columns:
        mae, rmse, r2, bias = calculate_metrics(data["actual"], data[col])
        print(f"{col:22s} MAE {mae:.3f}  RMSE {rmse:.2f}  R2 {r2:.3f}  bias {bias:.2f}")

----- all rows | n = 2229994
lgbm_predicted         MAE 24.692  RMSE 42.87  R2 0.809  bias 1.14
lgbm_fire_predicted    MAE 24.654  RMSE 42.85  R2 0.809  bias 0.96
----- actual < 100 | n = 1441564
lgbm_predicted         MAE 14.914  RMSE 23.65  R2 0.046  bias 7.02
lgbm_fire_predicted    MAE 14.785  RMSE 23.50  R2 0.058  bias 6.75
----- actual >= 100 | n = 788430
lgbm_predicted         MAE 42.570  RMSE 64.62  R2 0.606  bias -9.60
lgbm_fire_predicted    MAE 42.698  RMSE 64.68  R2 0.605  bias -9.61
----- Oct 15-Nov 30 2025, normal LOSO | n = 36426
lgbm_predicted         MAE 43.886  RMSE 68.94  R2 0.610  bias 2.66
lgbm_fire_predicted    MAE 44.014  RMSE 68.81  R2 0.611  bias 2.00
----- Oct 15-Nov 30 2025, window removed from training | n = 36426
lgbm_predicted         MAE 43.944  RMSE 69.17  R2 0.607  bias 2.83
lgbm_fire_predicted    MAE 43.566  RMSE 68.64  R2 0.613  bias 2.26


In [32]:
loso_df["lgbm_sq_error"] = (loso_df["actual"] - loso_df["lgbm_predicted"]) ** 2

total_sq_error = loso_df["lgbm_sq_error"].sum()
total_rows = len(loso_df)

share_df = loso_df.groupby("station_id")["lgbm_sq_error"].agg(["size", "sum"]).reset_index()
share_df["pct_of_rows"] = 100 * share_df["size"] / total_rows
share_df["pct_of_sq_error"] = 100 * share_df["sum"] / total_sq_error
share_df = share_df.drop(columns="sum").rename(columns={"size": "n_rows"})

print(share_df.sort_values("pct_of_sq_error", ascending=False).round(2).to_string())

                            station_id  n_rows  pct_of_rows  pct_of_sq_error
10                ihbas_dilshad_garden   67578         3.03             8.43
8                                  dtu   66459         2.98             6.13
28                            shadipur   68763         3.08             5.65
21                                nsut   69256         3.11             5.08
19                         nehru_nagar   69743         3.13             5.07
12                        jahangirpuri   68557         3.07             4.70
24                        punjabi_bagh   68883         3.09             4.08
1                          anand_vihar   65681         2.95             3.97
17                           najafgarh   67058         3.01             3.70
16                              mundka   65579         2.94             3.70
11                                 ito   66689         2.99             3.46
26                            rk_puram   67169         3.01             3.40

In [33]:
coords = df.groupby("station_id")[["latitude", "longitude"]].first()
names = coords.index.tolist()
lat = np.radians(coords["latitude"].to_numpy())
lon = np.radians(coords["longitude"].to_numpy())

a = (np.sin((lat[None, :] - lat[:, None]) / 2) ** 2
     + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin((lon[None, :] - lon[:, None]) / 2) ** 2)
D = 2 * 6371.0 * np.arcsin(np.sqrt(a))

# one row per pair, each pair listed once
i, j = np.triu_indices(len(names), k=1)
pairs = pd.DataFrame({
    "station_1": [names[x] for x in i],
    "station_2": [names[y] for y in j],
    "distance_km": D[i, j].round(2),
}).sort_values("distance_km").reset_index(drop=True)

pairs.to_csv("/kaggle/working/station_pairs_distance.csv", index=False)
print(pairs.to_string(index=False))

                         station_1                          station_2  distance_km
                              pusa                           shadipur         1.33
              ihbas_dilshad_garden                        vivek_vihar         1.59
                       ashok_vihar                           wazirpur         1.65
                               ito major_dhyan_chand_national_stadium         1.94
                               dtu                             rohini         2.13
          jawaharlal_nehru_stadium                        nehru_nagar         2.14
                 crri_mathura_road                      okhla_phase_2         2.28
                       anand_vihar                        vivek_vihar         2.84
                 crri_mathura_road                        nehru_nagar         2.92
                      punjabi_bagh                           shadipur         2.96
                          rk_puram                           sirifort         3.17
    

In [24]:
swapped = set(zip(pairs["station_2"], pairs["station_1"]))
print("rows:", len(pairs), "| expected:", 33 * 32 // 2)
print("reversed duplicates:", sum((a, b) in swapped for a, b in zip(pairs["station_1"], pairs["station_2"])))

rows: 528 | expected: 528
reversed duplicates: 0
